In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense,SimpleRNN,Embedding,Input

2025-08-10 11:32:02.599046: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-10 11:32:02.624994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754825522.644870    2187 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754825522.652251    2187 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754825522.682831    2187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
max_features = 10000
(X_train,y_train),(X_test,y_test) = imdb.load_data(num_words=max_features)

In [3]:
print(X_train.shape)
mx=0
for reveiw in X_train:
    mx = max(mx,len(reveiw))

max_len = mx+50+6
print(max_len)

(25000,)
2550


In [4]:
X_train = sequence.pad_sequences(X_train,maxlen=max_len)
X_test = sequence.pad_sequences(X_test,maxlen=max_len)

In [5]:
X_test[0].shape

(2550,)

In [12]:
inputs = Input(shape=(max_len,))
x = Embedding(max_features,128)(inputs)
x = SimpleRNN(128,activation='relu')(x)
outputs = Dense(1,activation='sigmoid')(x)

model = Model(inputs=inputs,outputs=outputs)

In [7]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 2550)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 2550, 128)      │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [14]:
from tensorflow.keras.callbacks import EarlyStopping

earlystop = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
earlystop

In [15]:
history = model.fit(X_train,y_train,epochs=10,batch_size=128,
                    validation_split=0.2,
                    callbacks=[earlystop]
)

Epoch 1/10


2025-08-10 11:43:26.915344: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 124 bytes spill stores, 124 bytes spill loads

2025-08-10 11:43:27.421808: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_34', 424 bytes spill stores, 424 bytes spill loads



157/157 ━━━━━━━━━━━━━━━━━━━━ 61s 361ms/step - accuracy: 0.5809 - loss: 19.5519 - val_accuracy: 0.5966 - val_loss: 0.6785
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 56s 357ms/step - accuracy: 0.6461 - loss: 0.6134 - val_accuracy: 0.6494 - val_loss: 0.6093
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 51s 323ms/step - accuracy: 0.7876 - loss: 0.4611 - val_accuracy: 0.7958 - val_loss: 0.4510
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 52s 333ms/step - accuracy: 0.8557 - loss: 0.3450 - val_accuracy: 0.7326 - val_loss: 0.5342
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 56s 355ms/step - accuracy: 0.8780 - loss: 0.3096 - val_accuracy: 0.8084 - val_loss: 0.4329
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 54s 346ms/step - accuracy: 0.9075 - loss: 0.2323 - val_accuracy: 0.8142 - val_loss: 0.4355
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 52s 333ms/step - accuracy: 0.9276 - loss: 0.1895 - val_accuracy: 0.8156 - val_loss: 0.4454
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 53s 340ms/step - accuracy: 0.9424 - loss: 0.1596 - va

## Prediction

In [19]:
word_index = imdb.get_word_index()
reverse_word_index = {value:key for key,value in word_index.items()}

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [23]:
def preprocess_review(review):
    words = review.lower().split()
    encoded_review = [word_index.get(word,2)+3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review],maxlen=max_len)
    return padded_review

In [26]:
def predict(review):
    preprocessed_review = preprocess_review(review)
    prediction = model.predict(preprocessed_review)[0][0]
    sentiment = "positive" if prediction > 0.5 else "negative"
    return sentiment,prediction

In [36]:
review = "It all began with an uplifting sense of possibility, the kind that makes you believe something special is about to unfold. But as things progressed, the charm faded into an endless stream of half-baked ideas, missed opportunities, and disappointments that piled up faster than they could be ignored. Then, just as the whole ordeal seemed ready to collapse under its own weight, a sudden burst of brilliance lit up the end, leaving you strangely satisfied despite everything in between"


predict(review)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 765ms/step


('positive', np.float32(0.88909155))